# NuIntNN Data Reader

Reads the output of `NuIntNNDataPrep_module` from `training_data.root`.

The module writes **4 TTrees** in separate subdirectories:

| Path | Contents |
|------|----------|
| `labels/tree` | Event ID, MC truth, training labels (dEprom, dEdir, dEspread, PCA) |
| `temporal/tree` | Temporal sequence features per flash |
| `ophits/tree` | Per-flash optical hit PE/channel/time |
| `images/tree` | 2D PMT images (uncoated/coated) + pe_per_channel |

In [ ]:
import uproot
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [ ]:
FILE_PATH = "training_data.root"  # update path as needed

print(f"File: {FILE_PATH}")
print(f"Exists: {Path(FILE_PATH).exists()}")
if Path(FILE_PATH).exists():
    print(f"Size: {Path(FILE_PATH).stat().st_size / 1024:.1f} KB")

f = uproot.open(FILE_PATH)
print("\nContents:")
for k in f.keys():
    print(f"  {k}")

## Labels tree

In [ ]:
labels = f["labels/tree"].arrays(library="pd")
print(f"Events total:         {len(labels)}")
print(f"Passed filters:       {labels['passed_filters'].sum()}")
print(f"TPC 0 / TPC 1:        {(labels['selected_tpc']==0).sum()} / {(labels['selected_tpc']==1).sum()}")
display(labels.head(10).round(2))

In [ ]:
passed = labels[labels["passed_filters"]]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Energy deposit centroid (dEprom) — passed events")
coords = [("dEprom_x", "X [cm]"), ("dEprom_y", "Y [cm]"), ("dEprom_z", "Z [cm]")]
for ax, (col, xlabel) in zip(axes, coords):
    ax.hist(passed[col].explode(), bins=50, alpha=0.7)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Events")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Direction label (dEdir) — passed events")
for ax, (col, xlabel) in zip(axes, [("dEdir_x","X"),("dEdir_y","Y"),("dEdir_z","Z")]):
    ax.hist(passed[col].explode(), bins=50, alpha=0.7, color="orange")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Events")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Images tree

In [ ]:
img_tree = f["images/tree"]
print(f"Image tree entries: {img_tree.num_entries}")

ny = img_tree["image_ny"].array(library="np")
nz = img_tree["image_nz"].array(library="np")
max_u = img_tree["max_pe_uncoated"].array(library="np")
max_c = img_tree["max_pe_coated"].array(library="np")
print(f"Image dimensions: ny={ny[0]}, nz={nz[0]} (first event)")

img_u_flat = img_tree["image_uncoated"].array(library="np")
img_c_flat = img_tree["image_coated"].array(library="np")

N = len(img_u_flat)
images_u = np.stack([img_u_flat[i].reshape(ny[i], nz[i]) for i in range(N)])
images_c = np.stack([img_c_flat[i].reshape(ny[i], nz[i]) for i in range(N)])
images = np.stack([images_u, images_c], axis=-1)  # (N, ny, nz, 2)

print(f"Images array shape: {images.shape}  (N, ny, nz, 2)")
print(f"max_pe_uncoated range: {max_u.min():.1f} – {max_u.max():.1f}")
print(f"max_pe_coated   range: {max_c.min():.1f} – {max_c.max():.1f}")

In [ ]:
def plot_event_images(images, idx, max_u, max_c):
    uncoated = images[idx, :, :, 0]
    coated   = images[idx, :, :, 1]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Event index {idx} — raw PE (unnormalized)", fontsize=13)
    for ax, data, title, mx in zip(
        axes,
        [uncoated, coated],
        [f"Uncoated  (max_pe={max_u[idx]:.1f})", f"Coated  (max_pe={max_c[idx]:.1f})"],
        [max_u[idx], max_c[idx]]
    ):
        im = ax.imshow(data, cmap="viridis", aspect="auto", interpolation="nearest",
                       vmin=0, vmax=mx if mx > 0 else 1)
        ax.set_title(title)
        ax.set_xlabel("Z bins")
        ax.set_ylabel("Y bins")
        plt.colorbar(im, ax=ax, label="PE")
    plt.tight_layout()
    plt.show()

CHOSEN_EVENT = 0  # change as needed
plot_event_images(images, idx=CHOSEN_EVENT, max_u=max_u, max_c=max_c)

## PE per channel

In [ ]:
pe_per_ch = img_tree["pe_per_channel"].array(library="np")
pe_arr = np.stack(pe_per_ch)  # (N, 312)
mean_pe = pe_arr.mean(axis=0)
print(f"pe_per_channel shape: {pe_arr.shape}")

plt.figure(figsize=(14, 4))
plt.bar(range(len(mean_pe)), mean_pe, width=1.0)
plt.xlabel("Channel index")
plt.ylabel("Mean PE")
plt.title("Mean PE per channel across all events")
plt.axvline(156, color='r', linestyle='--', label='TPC boundary (~ch 156)')
plt.legend()
plt.tight_layout()
plt.show()

## Ophits tree

In [ ]:
ophits = f["ophits/tree"]
flash_pe   = ophits["flash_ophit_pe"].array(library="ak")
flash_time = ophits["flash_ophit_time"].array(library="ak")

all_pe   = ak.to_numpy(ak.flatten(ak.flatten(flash_pe)))
all_time = ak.to_numpy(ak.flatten(ak.flatten(flash_time)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(all_pe[all_pe > 0], bins=80, log=True)
axes[0].set_xlabel("OpHit PE")
axes[0].set_ylabel("Count (log)")
axes[0].set_title("OpHit PE distribution")
axes[0].grid(True, alpha=0.3)

axes[1].hist(all_time, bins=80)
axes[1].set_xlabel("OpHit time [µs]")
axes[1].set_ylabel("Count")
axes[1].set_title("OpHit time distribution")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Temporal tree

In [ ]:
temporal = f["temporal/tree"]
seq_len  = temporal["temporal_seq_len"].array(library="np")
features = temporal["temporal_features"].array(library="np")
mask     = temporal["temporal_mask"].array(library="np")

print(f"Sequence lengths — min: {seq_len.min()}, max: {seq_len.max()}, mean: {seq_len.mean():.1f}")
print(f"Features shape (first event): {features[0].shape}")

plt.figure(figsize=(8, 4))
plt.hist(seq_len, bins=30)
plt.xlabel("Number of flashes (sequence length)")
plt.ylabel("Events")
plt.title("Temporal sequence length distribution")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()